# Identity Resolution: Goodreads, Amazon, DBpedia (Phase II)

**Entity = edition.** Two records match if they describe the same edition of a book
(same publication, format and ISBN), not just the same work. Different editions of the
same novel ("Where the Crawdads Sing" hardcover vs. large print) are *non-matches*.

Workflow, following the PyDI exercise (WDI05):
1. Load the three translated XML files and add cleaned helper columns
2. Gold standard: train / validation / test per dataset pair
3. Blocking: combine author-surname and title keys, evaluate pair completeness and reduction ratio
4. Features: PyDI comparators + "both values present" indicators
5. Matchers: simple rule-based baseline and ML models (selected on validation)
6. Apply the best model to all candidates, enforce one-to-one matching, evaluate on the test set
7. Save correspondences for data fusion

The ISBN is **not** used anywhere in matching; it was only used to build the gold standard.

## 1. Setup

In [1]:
import logging
import os
import re
import time
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import display

os.makedirs("logs", exist_ok=True)
logging.basicConfig(level=logging.INFO, format="[%(levelname)-5s] %(name)s - %(message)s",
                    handlers=[logging.FileHandler("logs/identity_resolution.log", encoding="utf-8")], force=True)

from PyDI.io import load_xml
from PyDI.entitymatching import FeatureExtractor, MLBasedMatcher, RuleBasedMatcher, GreedyOneToOneMatchingAlgorithm
from PyDI.entitymatching.base import BaseComparator
from PyDI.entitymatching.comparators import StringComparator, NumericComparator
from PyDI.entitymatching.evaluation import EntityMatchingEvaluator

DATA_DIR = Path("data")
GS_DIR = Path("gold_standard")
OUT_DIR = DATA_DIR / "output" / "identity_resolution"
CORR_DIR = DATA_DIR / "correspondences"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CORR_DIR.mkdir(parents=True, exist_ok=True)

PAIRS = [("goodreads", "amazon"), ("goodreads", "dbpedia"), ("amazon", "dbpedia")]
SEED = 42

## 2. Load data and add cleaned helper columns

The matchers compare the original attributes plus a few cleaned versions:
* `title_clean`: lower case, without bracket text such as "(Movie Tie-In)" or DBpedia's "(novel)",
  without subtitle and leading article
* `title_after_colon`: the part after a series name, e.g. "The Lost Fleet: Dauntless" -> "dauntless"
* `first_author`, `author_surname`: handles "(Goodreads Author)", "Vance, J. D." and "Jr."
* numeric `year`, `page_count`, `rating`, `ratings_count`, `price`

In [2]:
SUFFIXES = {"jr", "sr", "ii", "iii", "iv", "phd", "md", "dr", "ms", "mr", "mrs"}


def ascii_lower(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower() if isinstance(s, str) else ""


def clean_title(t):
    t = ascii_lower(t)
    t = re.sub(r"\([^)]*\)|\[[^\]]*\]", " ", t)
    t = re.sub(r"\s*(:| - ).*$", "", t)
    t = re.sub(r"[^a-z0-9 ]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return re.sub(r"^(the|a|an) ", "", t)


def first_author(a):
    if isinstance(a, list):
        a = a[0] if a else ""
    return re.sub(r"\(.*?\)", "", a).strip() if isinstance(a, str) else ""


def surname(a):
    a = re.sub(r"\(.*?\)", "", a or "").strip()
    if "," in a:
        w = re.sub(r"[^a-z ]", " ", ascii_lower(a.split(",")[0])).split()
        return w[-1] if w else ""
    w = [x for x in re.sub(r"[^a-z ]", " ", ascii_lower(a)).split() if x not in SUFFIXES]
    return w[-1] if w else ""


def load(name):
    df = load_xml(DATA_DIR / f"{name}.xml", name=name)
    df.attrs["source"] = name
    for col in ["title", "authors_author", "publisher", "publication_date", "page_count", "language",
                "format", "edition", "rating", "ratings_count", "price"]:
        if col not in df.columns:
            df[col] = None
    df["title_clean"] = df["title"].map(clean_title)
    df["title_after_colon"] = df["title"].map(
        lambda t: clean_title(t.split(":", 1)[1]) if isinstance(t, str) and ":" in t else clean_title(t))
    df["first_author"] = df["authors_author"].map(first_author)
    df["author_surname"] = df["first_author"].map(surname)
    df["publisher_clean"] = df["publisher"].map(ascii_lower)
    df["format_clean"] = df["format"].map(ascii_lower)
    df["language_clean"] = df["language"].map(ascii_lower)
    df["year"] = pd.to_numeric(df["publication_date"].astype("string").str[:4], errors="coerce")
    for c in ["page_count", "rating", "ratings_count", "price"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


data = {name: load(name) for name in ["goodreads", "amazon", "dbpedia"]}
for name, df in data.items():
    print(f"{name:<10} {len(df):>7,} records")
display(data["amazon"][["id", "title", "title_clean", "first_author", "author_surname", "publisher", "year", "page_count", "format"]].head())

goodreads   52,478 records
amazon      40,000 records
dbpedia     59,270 records


,id,title,title_clean,first_author,author_surname,publisher,year,page_count,format
0,amazon_id_1,Where the Crawdads Sing,where the crawdads sing,Delia Owens,owens,Penguin Audio,2019,NaN,NaN
1,amazon_id_2,Where the Crawdads Sing,where the crawdads sing,Delia Owens,owens,Corsair,2019,NaN,Hardcover
2,amazon_id_3,Where the Crawdads Sing,where the crawdads sing,Delia Owens,owens,Random House Large Print,2018,496.0,Paperback
3,amazon_id_4,Where the Crawdads Sing (Movie Tie-In),where the crawdads sing,Delia Owens,owens,G.P. Putnam's Sons,2022,400.0,Paperback
4,amazon_id_5,Where the Crawdads Sing,where the crawdads sing,Delia Owens,owens,Corsair,2019,NaN,Paperback


## 3. Gold standard: train / validation / test

`gold_standard/<pair>_train.csv` and `<pair>_test.csv` come from `build_gold_standard.py`
(ISBN-based, 539 pairs checked by hand, wrong pairs removed). We split 25% of the training
pairs off as a **validation set** (stratified, fixed seed) for choosing blocking, models and
thresholds. The **test set is only used once, for the final evaluation.**

In [3]:
from sklearn.model_selection import train_test_split


def read_gs(path):
    df = pd.read_csv(path)
    df["label"] = df["label"].astype(str).str.upper().eq("TRUE")
    return df[["id1", "id2", "label"]]


gold = {}
for a, b in PAIRS:
    p = f"{a}_{b}"
    train_full = read_gs(GS_DIR / f"{p}_train.csv")
    train, val = train_test_split(train_full, test_size=0.25, stratify=train_full["label"], random_state=SEED)
    gold[p] = {"train": train.reset_index(drop=True), "validation": val.reset_index(drop=True),
               "test": read_gs(GS_DIR / f"{p}_test.csv")}
    print(p, {k: f"{len(v)} ({v.label.sum()} TRUE)" for k, v in gold[p].items()})

goodreads_amazon {'train': '624 (201 TRUE)', 'validation': '209 (67 TRUE)', 'test': '357 (97 TRUE)'}
goodreads_dbpedia {'train': '633 (209 TRUE)', 'validation': '212 (70 TRUE)', 'test': '363 (108 TRUE)'}
amazon_dbpedia {'train': '597 (171 TRUE)', 'validation': '199 (57 TRUE)', 'test': '342 (89 TRUE)'}


## 4. Blocking

Comparing all pairs would mean 2–3 billion comparisons per dataset pair. We tested three
blocking keys on the training + validation pairs (target: pair completeness above 97%):

* **author surname** alone misses co-authors listed in a different order and pseudonyms
  (Richard Bachman / Stephen King, J.D. Robb / Nora Roberts)
* **cleaned title** alone misses Amazon/DBpedia titles with series prefixes or subtitles
* the **union** of surname, cleaned title and title-after-colon keys keeps almost all matches

Reduction ratio = share of all possible pairs that are *not* compared.

In [4]:
def key_pairs(A, B, col_a, col_b):
    a = A.loc[A[col_a] != "", ["id", col_a]].rename(columns={col_a: "k"})
    b = B.loc[B[col_b] != "", ["id", col_b]].rename(columns={col_b: "k"})
    return a.merge(b, on="k", suffixes=("1", "2"))[["id1", "id2"]]


BLOCKING_KEYS = {
    "surname": [("author_surname", "author_surname")],
    "title": [("title_clean", "title_clean")],
    "surname OR title (+ after colon)": [("author_surname", "author_surname"), ("title_clean", "title_clean"),
                                         ("title_clean", "title_after_colon"), ("title_after_colon", "title_clean")],
}


def block(A, B, keys):
    return pd.concat([key_pairs(A, B, ca, cb) for ca, cb in keys]).drop_duplicates().reset_index(drop=True)


def blocking_metrics(cands, gs_pairs, n_all):
    pos = set(zip(gs_pairs.loc[gs_pairs.label, "id1"], gs_pairs.loc[gs_pairs.label, "id2"]))
    found = len(pos & set(zip(cands.id1, cands.id2)))
    return {"candidates": len(cands), "pair completeness": found / len(pos), "reduction ratio": 1 - len(cands) / n_all}


rows, candidates = [], {}
for a, b in PAIRS:
    p = f"{a}_{b}"
    A, B = data[a], data[b]
    dev = pd.concat([gold[p]["train"], gold[p]["validation"]])
    for name, keys in BLOCKING_KEYS.items():
        t = time.time()
        c = block(A, B, keys)
        m = blocking_metrics(c, dev, len(A) * len(B))
        rows.append({"pair": p, "blocking": name, **m, "seconds": round(time.time() - t, 1)})
        if name.startswith("surname OR"):
            candidates[p] = c
            if os.environ.get("QUICK"):   # debugging only: small sample of candidates
                candidates[p] = c.sample(min(len(c), 30_000), random_state=SEED)
blocking_results = pd.DataFrame(rows)
blocking_results.to_csv(OUT_DIR / "blocking_results.csv", index=False)
display(blocking_results.style.format({"candidates": "{:,}", "pair completeness": "{:.1%}", "reduction ratio": "{:.4%}"}))

,pair,blocking,candidates,pair completeness,reduction ratio,seconds
0,goodreads_amazon,surname,"1,299,848",96.6%,99.9381%,2.500000
1,goodreads_amazon,title,"36,272",77.6%,99.9983%,0.100000
2,goodreads_amazon,surname OR title (+ after colon),"1,320,641",100.0%,99.9371%,2.000000
3,goodreads_dbpedia,surname,"1,062,280",92.1%,99.9658%,1.400000
4,goodreads_dbpedia,title,"21,194",58.8%,99.9993%,0.100000
5,goodreads_dbpedia,surname OR title (+ after colon),"1,074,254",98.2%,99.9655%,1.600000
6,amazon_dbpedia,surname,"1,200,724",92.1%,99.9494%,1.300000
7,amazon_dbpedia,title,"28,803",73.7%,99.9988%,0.100000
8,amazon_dbpedia,surname OR title (+ after colon),"1,216,635",97.4%,99.9487%,1.600000


## 5. Features

PyDI comparators on title, author, publisher, year, page count, format and language.
Missing values get similarity 0 in PyDI, which looks the same as "completely different".
Because DBpedia often lacks page count and format, we add **presence indicators**
(1 if both records have a value), so the models can tell "different" from "unknown".

In [5]:
class BothPresent(BaseComparator):
    """1.0 if both records have a value in `column`, else 0.0."""

    def __init__(self, column):
        super().__init__(name=f"BothPresent({column})")
        self.column = column

    def compare(self, record1, record2):
        def ok(v):
            return v is not None and not (isinstance(v, float) and pd.isna(v)) and str(v).strip() not in ("", "nan")
        return float(ok(record1.get(self.column)) and ok(record2.get(self.column)))


comparators = [
    StringComparator("title_clean", similarity_function="jaro_winkler"),
    StringComparator("title_clean", similarity_function="levenshtein"),
    StringComparator("title", similarity_function="jaccard", tokenization="word", preprocess=str.lower),
    StringComparator("title_after_colon", similarity_function="jaro_winkler"),
    StringComparator("first_author", similarity_function="jaro_winkler", preprocess=str.lower),
    StringComparator("authors_author", similarity_function="jaccard", tokenization="word",
                     preprocess=str.lower, list_strategy="best_match"),
    StringComparator("publisher_clean", similarity_function="jaccard", tokenization="word"),
    NumericComparator("year", method="absolute_difference", max_difference=10),
    NumericComparator("page_count", method="relative_difference", max_difference=0.5),
    StringComparator("format_clean", similarity_function="jaro_winkler"),
    StringComparator("language_clean", similarity_function="jaro_winkler"),
    BothPresent("year"), BothPresent("page_count"), BothPresent("publisher"), BothPresent("format"),
]
feature_extractor = FeatureExtractor(comparators)

features = {}
for a, b in PAIRS:
    p = f"{a}_{b}"
    features[p] = {split: feature_extractor.create_features(data[a], data[b], gs[["id1", "id2"]],
                                                            labels=gs["label"], id_column="id")
                   for split, gs in gold[p].items()}
FEATURE_COLUMNS = [c for c in features[f"{PAIRS[0][0]}_{PAIRS[0][1]}"]["train"].columns if c not in ("id1", "id2", "label")]
print(len(FEATURE_COLUMNS), "features")
display(features["goodreads_amazon"]["train"].head(3).T)

15 features


,0,1,2
id1,goodreads_id_494,goodreads_id_784,goodreads_id_41047
id2,amazon_id_9643,amazon_id_13424,amazon_id_25877
"StringComparator(title_clean, jaro_winkler, tokenization=char, list_strategy=None)",1.0,1.0,0.548622
"StringComparator(title_clean, levenshtein, tokenization=char, list_strategy=None)",1.0,1.0,0.105263
"StringComparator(title, jaccard, tokenization=word, list_strategy=None)",1.0,1.0,0.0
"StringComparator(title_after_colon, jaro_winkler, tokenization=char, list_strategy=None)",1.0,1.0,0.412698
"StringComparator(first_author, jaro_winkler, tokenization=char, list_strategy=None)",1.0,0.966667,0.545752
"StringComparator(authors_author, jaccard, tokenization=word, list_strategy=best_match)",0.5,0.666667,0.25
"StringComparator(publisher_clean, jaccard, tokenization=word, list_strategy=None)",0.0,0.333333,0.0
"NumericComparator(year, absolute_difference, list_strategy=None)",0.9,0.0,0.0


## 6. Matchers

### 6.1 Rule-based baseline
A simple weighted rule (title 0.5, author 0.3, year 0.2), with the threshold chosen on the
validation set. This is only a **baseline** to compare the ML models against; the tuned
rule-based matcher is developed separately.

In [6]:
from sklearn.metrics import f1_score, precision_score, recall_score

BASELINE = {"StringComparator(title_clean, jaro_winkler": 0.5, "StringComparator(first_author, jaro_winkler": 0.3,
            "NumericComparator(year, absolute_difference": 0.2}


def baseline_score(df):
    cols = {prefix: next(c for c in FEATURE_COLUMNS if c.startswith(prefix)) for prefix in BASELINE}
    return sum(df[cols[k]] * w for k, w in BASELINE.items())


def scores(y, pred):
    return {"precision": precision_score(y, pred, zero_division=0), "recall": recall_score(y, pred, zero_division=0),
            "f1": f1_score(y, pred, zero_division=0)}


baseline = {}
for p in features:
    val = features[p]["validation"]
    best = max(((t, f1_score(val.label, baseline_score(val) >= t)) for t in [x / 100 for x in range(50, 100)]),
               key=lambda x: x[1])
    baseline[p] = best[0]
    print(f"{p:<20} best threshold {best[0]:.2f} -> validation F1 {best[1]:.3f}")

goodreads_amazon     best threshold 0.99 -> validation F1 0.698


goodreads_dbpedia    best threshold 0.65 -> validation F1 0.611


amazon_dbpedia       best threshold 0.99 -> validation F1 0.584


### 6.2 ML-based matchers
Grid search with 5-fold cross-validation on the training set, for each dataset pair
separately (the attributes available differ between the pairs). The model with the best
**validation** F1 is selected.

In [7]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

MODELS = {
    "LogisticRegression": (LogisticRegression(max_iter=2000, random_state=SEED),
                           {"C": [0.1, 1.0, 10.0], "class_weight": [None, "balanced"]}),
    "RandomForest": (RandomForestClassifier(random_state=SEED),
                     {"n_estimators": [100, 300], "max_depth": [5, 10, None], "class_weight": [None, "balanced"]}),
    "GradientBoosting": (GradientBoostingClassifier(random_state=SEED),
                         {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1], "max_depth": [2, 3]}),
}

model_rows, best_models = [], {}
for p in features:
    tr, va = features[p]["train"], features[p]["validation"]
    best_f1 = -1
    for name, (model, grid) in MODELS.items():
        gs = GridSearchCV(model, grid, scoring="f1", cv=StratifiedKFold(5, shuffle=True, random_state=SEED), n_jobs=-1)
        gs.fit(tr[FEATURE_COLUMNS], tr["label"])
        val_scores = scores(va["label"], gs.best_estimator_.predict(va[FEATURE_COLUMNS]))
        model_rows.append({"pair": p, "model": name, "cv f1 (train)": gs.best_score_, **{f"val {k}": v for k, v in val_scores.items()},
                           "params": gs.best_params_})
        if val_scores["f1"] > best_f1:
            best_f1, best_models[p] = val_scores["f1"], (name, gs.best_estimator_)
    bl = scores(va["label"], baseline_score(va) >= baseline[p])
    model_rows.append({"pair": p, "model": "Rule baseline", "cv f1 (train)": None, **{f"val {k}": v for k, v in bl.items()},
                       "params": {"threshold": baseline[p]}})
model_results = pd.DataFrame(model_rows)
model_results.to_csv(OUT_DIR / "model_selection.csv", index=False)
display(model_results.style.format({c: "{:.3f}" for c in model_results.columns if "f1" in c or "precision" in c or "recall" in c}, na_rep="-"))
print({p: m[0] for p, m in best_models.items()})

,pair,model,cv f1 (train),val precision,val recall,val f1,params
0,goodreads_amazon,LogisticRegression,0.853,0.838,0.851,0.844,"{'C': 1.0, 'class_weight': None}"
1,goodreads_amazon,RandomForest,0.901,0.840,0.940,0.887,"{'class_weight': 'balanced', 'max_depth': 5, 'n_estimators': 100}"
2,goodreads_amazon,GradientBoosting,0.896,0.905,0.851,0.877,"{'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 100}"
3,goodreads_amazon,Rule baseline,-,0.746,0.657,0.698,{'threshold': 0.99}
4,goodreads_dbpedia,LogisticRegression,0.733,0.683,0.800,0.737,"{'C': 10.0, 'class_weight': 'balanced'}"
5,goodreads_dbpedia,RandomForest,0.819,0.728,0.843,0.781,"{'class_weight': 'balanced', 'max_depth': 10, 'n_estimators': 100}"
6,goodreads_dbpedia,GradientBoosting,0.802,0.733,0.786,0.759,"{'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 200}"
7,goodreads_dbpedia,Rule baseline,-,0.466,0.886,0.611,{'threshold': 0.65}
8,amazon_dbpedia,LogisticRegression,0.726,0.700,0.860,0.772,"{'C': 10.0, 'class_weight': 'balanced'}"
9,amazon_dbpedia,RandomForest,0.832,0.717,0.754,0.735,"{'class_weight': 'balanced', 'max_depth': 10, 'n_estimators': 300}"


{'goodreads_amazon': 'RandomForest', 'goodreads_dbpedia': 'RandomForest', 'amazon_dbpedia': 'LogisticRegression'}


In [8]:
# Feature importance of the selected models (tree models) / coefficients (logistic regression)
for p, (name, model) in best_models.items():
    imp = getattr(model, "feature_importances_", None)
    if imp is None:
        imp = abs(model.coef_[0])
    top = pd.Series(imp, index=FEATURE_COLUMNS).sort_values(ascending=False).head(6)
    print(f"\n{p} ({name})"); print(top.round(3).to_string())


goodreads_amazon (RandomForest)
NumericComparator(page_count, relative_difference, list_strategy=None)                 0.225
StringComparator(publisher_clean, jaccard, tokenization=word, list_strategy=None)      0.196
NumericComparator(year, absolute_difference, list_strategy=None)                       0.192
StringComparator(format_clean, jaro_winkler, tokenization=char, list_strategy=None)    0.130
StringComparator(title, jaccard, tokenization=word, list_strategy=None)                0.074
StringComparator(title_clean, jaro_winkler, tokenization=char, list_strategy=None)     0.045

goodreads_dbpedia (RandomForest)
NumericComparator(year, absolute_difference, list_strategy=None)                            0.213
NumericComparator(page_count, relative_difference, list_strategy=None)                      0.160
StringComparator(publisher_clean, jaccard, tokenization=word, list_strategy=None)           0.159
StringComparator(title_after_colon, jaro_winkler, tokenization=char, list_strateg

## 7. Apply to all candidates, one-to-one matching, test evaluation

The selected model is applied to all candidate pairs from blocking (score = match probability,
threshold 0.5). Because each source should contain every edition at most once, we also try a
**one-to-one constraint** with greedy global matching. Whether to use it is decided on the
**validation** set; the test set is only reported. Matches lost in blocking count as false
negatives, so the results include the blocking loss.

In [9]:
def chunks(df, size=100_000):
    for i in range(0, len(df), size):
        yield df.iloc[i:i + size]


results, correspondences = [], {}
for a, b in PAIRS:
    p = f"{a}_{b}"
    name, model = best_models[p]
    t = time.time()
    # use_probabilities=True: the score is the predicted match probability. Without it every
    # match gets score 1.0, and one-to-one matching would pick an arbitrary edition.
    corr = MLBasedMatcher(feature_extractor).match(data[a], data[b], candidates=list(chunks(candidates[p])),
                                                   id_column="id", trained_classifier=model,
                                                   use_probabilities=True, threshold=0.5)
    minutes = (time.time() - t) / 60

    # Each source lists every edition at most once, so we enforce one-to-one matching.
    # The match threshold is chosen on the VALIDATION set (ties -> stricter threshold);
    # the variant without one-to-one is only reported for comparison.
    def evaluate(c, split, tag):
        ev = EntityMatchingEvaluator.evaluate_matching(correspondences=c, test_pairs=gold[p][split],
                                                       out_dir=OUT_DIR / f"eval_{p}_{tag}_{split}")
        return ev["precision"], ev["recall"], ev["f1"]

    for split in ["validation", "test"]:
        P, R, F = evaluate(corr, split, "without_11")
        results.append({"pair": p, "model": name, "variant": "threshold 0.5, without 1:1", "split": split,
                        "correspondences": len(corr), "precision": P, "recall": R, "f1": F, "minutes": round(minutes, 1)})
    best = None
    for thr in [0.5, 0.6, 0.7, 0.8, 0.9]:
        c = GreedyOneToOneMatchingAlgorithm().cluster(corr[corr["score"] >= thr].reset_index(drop=True))
        P, R, F = evaluate(c, "validation", f"t{int(thr * 10)}_11")
        results.append({"pair": p, "model": name, "variant": f"threshold {thr}, with 1:1", "split": "validation",
                        "correspondences": len(c), "precision": P, "recall": R, "f1": F, "minutes": round(minutes, 1)})
        if best is None or F >= best[1]:
            best = (thr, F, c)
    thr, _, chosen = best
    P, R, F = evaluate(chosen, "test", f"t{int(thr * 10)}_11")
    results.append({"pair": p, "model": name, "variant": f"threshold {thr}, with 1:1 (selected)", "split": "test",
                    "correspondences": len(chosen), "precision": P, "recall": R, "f1": F, "minutes": round(minutes, 1)})
    correspondences[p] = chosen
    print(f"{p}: threshold {thr} chosen on validation -> {len(chosen):,} correspondences, test F1 {F:.3f}")
test_results = pd.DataFrame(results)
test_results.to_csv(OUT_DIR / "test_results.csv", index=False)
display(test_results.style.format({"precision": "{:.3f}", "recall": "{:.3f}", "f1": "{:.3f}", "correspondences": "{:,}"}))

goodreads_amazon: threshold 0.5 chosen on validation -> 3,898 correspondences, test F1 0.874


goodreads_dbpedia: threshold 0.5 chosen on validation -> 8,058 correspondences, test F1 0.725


amazon_dbpedia: threshold 0.6 chosen on validation -> 3,684 correspondences, test F1 0.676


,pair,model,variant,split,correspondences,precision,recall,f1,minutes
0,goodreads_amazon,RandomForest,"threshold 0.5, without 1:1",validation,"13,023",0.840,0.940,0.887,8.700000
1,goodreads_amazon,RandomForest,"threshold 0.5, without 1:1",test,"13,023",0.806,0.897,0.849,8.700000
2,goodreads_amazon,RandomForest,"threshold 0.5, with 1:1",validation,"3,898",1.000,0.746,0.855,8.700000
3,goodreads_amazon,RandomForest,"threshold 0.6, with 1:1",validation,"3,193",1.000,0.731,0.845,8.700000
4,goodreads_amazon,RandomForest,"threshold 0.7, with 1:1",validation,"2,532",1.000,0.687,0.814,8.700000
5,goodreads_amazon,RandomForest,"threshold 0.8, with 1:1",validation,"2,172",1.000,0.642,0.782,8.700000
6,goodreads_amazon,RandomForest,"threshold 0.9, with 1:1",validation,"1,387",1.000,0.493,0.660,8.700000
7,goodreads_amazon,RandomForest,"threshold 0.5, with 1:1 (selected)",test,"3,898",0.987,0.784,0.874,8.700000
8,goodreads_dbpedia,RandomForest,"threshold 0.5, without 1:1",validation,"21,335",0.722,0.814,0.765,7.200000
9,goodreads_dbpedia,RandomForest,"threshold 0.5, without 1:1",test,"21,335",0.732,0.759,0.745,7.200000


### Plausibility check
The gold standard only contains ~1,200 pairs per dataset pair, so wrong correspondences *outside* it do not lower the scores (lecture: "your evaluation is only as good as your evaluation set"). As a sanity check we compare the number of correspondences with a rough estimate of the true overlap: shared ISBNs divided by the share of records that have an ISBN on both sides.

In [10]:
from overlap_estimate import load_goodreads, load_dbpedia, load_amazon, isbn_set

isbn_data = {"goodreads": load_goodreads(), "amazon": load_amazon("data/amazon_books_subset.csv"), "dbpedia": load_dbpedia()}
rows = []
for a, b in PAIRS:
    A, B = isbn_data[a], isbn_data[b]
    shared = len(isbn_set(A) & isbn_set(B))
    cover = (A.isbns.str.len() > 0).mean() * (B.isbns.str.len() > 0).mean()
    rows.append({"pair": f"{a}_{b}", "shared ISBNs": shared, "estimated true matches": round(shared / cover),
                 "correspondences without 1:1": int(test_results[(test_results.pair == f"{a}_{b}") & test_results.variant.str.contains("without")].correspondences.iloc[0]),
                 "final correspondences": len(correspondences[f"{a}_{b}"])})
display(pd.DataFrame(rows))

,pair,shared ISBNs,estimated true matches,correspondences without 1:1,final correspondences
0,goodreads_amazon,2169,2642,13023,3898
1,goodreads_dbpedia,3271,6590,21335,8058
2,amazon_dbpedia,1725,2854,40698,3684


### Cluster size distribution
With one-to-one matching inside each pair, larger clusters can only come from combining the
three pairs. Many clusters larger than 3 would point to wrong correspondences.

In [11]:
all_corr = pd.concat(correspondences.values(), ignore_index=True)
dist = EntityMatchingEvaluator.create_cluster_size_distribution(correspondences=all_corr, out_dir=OUT_DIR / "cluster_analysis")
display(dist)

,cluster_size,frequency,percentage
0,2,10209,83.454590
1,3,1359,11.109295
2,4,536,4.381591
3,5,62,0.506826
4,6,35,0.286111
5,7,21,0.171667
6,8,5,0.040873
7,9,3,0.024524
8,10,3,0.024524


## 8. Error analysis (test set)
False positives and false negatives of the final correspondences, with the main attributes.

In [12]:
def error_table(p, n=8):
    a, b = p.split("_")
    A, B = data[a].set_index("id"), data[b].set_index("id")
    pred = set(zip(correspondences[p].id1, correspondences[p].id2))
    te = gold[p]["test"]
    te = te.assign(pred=[(x, y) in pred for x, y in zip(te.id1, te.id2)])
    cols = ["title", "first_author", "publisher", "year", "page_count", "format"]
    out = []
    for kind, mask in [("false positive", te.pred & ~te.label), ("false negative", ~te.pred & te.label)]:
        for x, y in zip(te.loc[mask, "id1"].head(n), te.loc[mask, "id2"].head(n)):
            out.append({"error": kind, **{f"{c} 1": A.at[x, c] for c in cols}, **{f"{c} 2": B.at[y, c] for c in cols}})
    return pd.DataFrame(out)


for p in correspondences:
    print(p)
    display(error_table(p))

goodreads_amazon


,error,title 1,first_author 1,publisher 1,year 1,page_count 1,format 1,title 2,first_author 2,publisher 2,year 2,page_count 2,format 2
0,false positive,Two for the Dough,Janet Evanovich,Penguin Books,<NA>,326.0,Paperback,"Two for the Dough (Stephanie Plum, No. 2) (Ste...",Janet Evanovich,Pocket Books,1996,336.0,Paperback
1,false negative,Stormbreaker,Anthony Horowitz,Speak,2004,256.0,Paperback,Stormbreaker (Alex Rider),Anthony Horowitz,Penguin,2004,256.0,Paperback
2,false negative,Maybe in Another Life,Taylor Jenkins Reid,"Dreamscape Media, LLC",2015,10.0,Audiobook,Maybe in Another Life: A Novel,Taylor Jenkins Reid,Washington Square Press,2015,352.0,Paperback
3,false negative,Rich People Problems,Kevin Kwan,Doubleday,2017,398.0,Hardcover,Rich People Problems (Crazy Rich Asians Trilogy),Kevin Kwan,Doubleday,2017,416.0,Hardcover
4,false negative,The Stranger in the Woods: The Extraordinary S...,Michael Finkel,Books on Tape,2017,224.0,Audio CD,The Stranger in the Woods,Michael Finkel,BooksOnTape,2017,NaN,Audio CD
5,false negative,"Oh, the Thinks You Can Think!",Dr. Seuss,Beginner Books,1975,64.0,Paperback,"Oh, the Thinks You Can Think! Green Back Book",Dr. Seuss,Harpercollins Childs,2004,64.0,Paperback
6,false negative,Now You See Her,James Patterson,"Little, Brown and Company",2011,359.0,Mass Market Paperback,Now You See Her,James Patterson,"Little, Brown and Company",2011,432.0,Hardcover
7,false negative,It,Stephen King,New English Library,<NA>,1116.0,Paperback,It,Stephen King,Trafalgar Square,1987,920.0,Paperback
8,false negative,Chicka Chicka Boom Boom,Bill Martin Jr.,Beach Lane Books,2000,40.0,Paperback,Chicka Chicka Boom Boom,John Archambault,Beach Lane Books,2000,32.0,Paperback


goodreads_dbpedia


,error,title 1,first_author 1,publisher 1,year 1,page_count 1,format 1,title 2,first_author 2,publisher 2,year 2,page_count 2,format 2
0,false positive,Olvidado rey Gudú,Ana María Matute,Espasa Calpe,1996,872.0,Hardcover,Olvidado rey Gudú,Ana María Matute,Espasa-Calpe,1996,865.0,None
1,false positive,Reservation Blues,Sherman Alexie,Grove Press,2005,306.0,Paperback,Reservation Blues,Sherman Alexie,NaN,1995,306.0,None
2,false positive,Oscar and Lucinda,Peter Carey,University of Queensland Press,1998,515.0,Paperback,Oscar and Lucinda,Peter Carey,University of Queensland Press,1988,528.0,None
3,false positive,The Prophecy,Chris Kuzneski,Penguin,2009,547.0,Paperback,The Prophecy (Kuzneski novel),Chris Kuzneski,G. P. Putnam's Sons,2009,NaN,None
4,false positive,Death of Kings,Bernard Cornwell,HarperCollins Publishers,2011,335.0,Hardcover,Death of Kings,Bernard Cornwell,HarperCollins,2011,335.0,None
5,false positive,Tietam Brown,Mick Foley,Knopf,2003,256.0,Hardcover,Tietam Brown,Mick Foley,Alfred A. Knopf,2003,243.0,None
6,false positive,Sword of God,Chris Kuzneski,Penguin,2007,432.0,Paperback,Sword of God (novel),Chris Kuzneski,Penguin Books,2007,NaN,None
7,false positive,Drift: The Unmooring of American Military Power,Rachel Maddow,Random House Audio,2012,288.0,NaN,Drift: The Unmooring of American Military Power,Rachel Maddow,Crown Publishing Group,2012,288.0,None
8,false negative,Enrique's Journey,Sonia Nazario,Random House Trade Paperbacks,2007,294.0,Paperback,Enrique's Journey,,skid,<NA>,NaN,None
9,false negative,Timeless,Alexandra Monir,Delacorte Press,2011,280.0,Hardcover,Timeless (Monir novel),Alexandra Monir,Random House Children's Books,<NA>,304.0,None


amazon_dbpedia


,error,title 1,first_author 1,publisher 1,year 1,page_count 1,format 1,title 2,first_author 2,publisher 2,year 2,page_count 2,format 2
0,false positive,"Even Dogs in the Wild (A Rebus Novel, 20)",Ian Rankin,"Little, Brown and Company",2016,352.0,Hardcover,Even Dogs in the Wild,Ian Rankin,Orion Publishing Group,2015,352.0,None
1,false positive,Life,James Fox,"Little, Brown and Company",2010,576.0,Hardcover,Life (Richards book),James Fox,"Little, Brown and Company",2010,564.0,None
2,false positive,Scythe (1) (Arc of a Scythe),Neal Shusterman,Simon & Schuster Books for Young Readers,2016,448.0,Hardcover,Scythe (novel),Neal Shusterman,Simon & Schuster,2016,443.0,None
3,false positive,"Where'd You Go, Bernadette: A Novel",Maria Semple,"Little, Brown and Company",2012,336.0,Hardcover,"Where'd You Go, Bernadette",Maria Semple,"Little, Brown and Company",2012,336.0,None
4,false positive,The Wind-Up Bird Chronicle,Haruki Murakami,Knopf,1997,611.0,Hardcover,The Wind-Up Bird Chronicle,Haruki Murakami,NaN,1997,607.0,None
5,false positive,"The Happiness Project (Revised Edition): Or, W...",Gretchen Rubin,Harper Paperbacks,2015,368.0,Paperback,"Nazis, Islamists, and the Making of the Modern...",Barry Rubin,Yale University Press,2014,360.0,None
6,false positive,The Night Manager,John le Carré,Ballantine Books,1994,443.0,Paperback,The Night Manager,John le Carré,Hodder & Stoughton,1993,443.0,None
7,false positive,Team of Rivals,Doris Kearns Goodwin,Simon & Schuster,2005,944.0,Hardcover,Team of Rivals,Doris Kearns Goodwin,Simon & Schuster,2005,944.0,None
8,false negative,Warriors: Into the Wild,Erin Hunter,HarperCollins,2003,288.0,Hardcover,Into the Wild (novel),Erin Hunter,Avon (publisher),2003,272.0,None
9,false negative,A Land Remembered,Patrick D. Smith,Pineapple Press,1998,416.0,Hardcover,A Land Remembered,Patrick D. Smith,Pineapple Press,1984,403.0,None


## 9. Save correspondences for data fusion
One CSV per dataset pair (`id1, id2, score`) in `data/correspondences/`.

In [13]:
for p, c in correspondences.items():
    c.to_csv(CORR_DIR / f"{p}.csv", index=False)
    print(f"{p:<20} {len(c):>6,} correspondences -> {CORR_DIR / (p + '.csv')}")

goodreads_amazon      3,898 correspondences -> data/correspondences/goodreads_amazon.csv
goodreads_dbpedia     8,058 correspondences -> data/correspondences/goodreads_dbpedia.csv
amazon_dbpedia        3,684 correspondences -> data/correspondences/amazon_dbpedia.csv
